In [7]:

import os
import asyncio
import json
import sys
import pathlib



from dotenv import load_dotenv
load_dotenv()  # 读取 backend/.env

from loguru import logger
from agent.task_queue import (
    enqueue_task,
    start_worker,
    read_task_events,
    TASK_STREAM,
)
from agent.logger import setup_logger
from agent.task_queue import _process_one_task, _get_redis, _ensure_consumer_group
import redis.asyncio as redis

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
TASK_STREAM = "research:tasks"
EVENT_STREAM_PREFIX = "research:events"
CONSUMER_GROUP = "research-workers"
# 单个任务的事件流最大保留条数，防止无限增长
EVENT_STREAM_MAXLEN = 500
# XREAD 阻塞超时（毫秒），避免空轮询
STREAM_BLOCK_MS = 5000
setup_logger()


<loguru.logger handlers=[(id=7, level=20, sink=stderr), (id=8, level=10, sink='logs/ZhiPoAI_DR_{time:YYYY-MM-DD}.log')]>

In [4]:
task_id = '5c4e8aee-8de5-4f32-85e6-6a42ffaab7fc'

In [14]:
async def consumer_group(task_id: str):
    """模拟 start_worker 的核心循环 ——
    从 Redis Stream 消费一条任务 → 执行 graph → 把事件写回 event stream。
    """
    print("\n" + "=" * 70)
    print(f"  [Step 2] 模拟 Worker 消费 task_id={task_id[:8]}...")
    print("=" * 70)

    r = await _get_redis()
    # await _ensure_consumer_group(r)  # 确保 consumer group 存在
    """创建 Consumer Group（如果不存在）."""
    try:
        await r.xgroup_create(
            TASK_STREAM,           # 参数1: 给哪个 Stream 建组
            CONSUMER_GROUP,        # 参数2: 组名叫什么
            id="0",                # 参数3: 从哪个消息 ID 开始读
            mkstream=True,         # 参数4: Stream 不存在时是否自动建
        )
        logger.info(f"[TaskQueue] Consumer Group '{CONSUMER_GROUP}' 已创建")
    except redis.ResponseError as e:
        if "BUSYGROUP" in str(e):
            logger.debug(f"[TaskQueue] Consumer Group 已存在，跳过创建")
        else:
            raise


await consumer_group(task_id)


  [Step 2] 模拟 Worker 消费 task_id=5c4e8aee...


In [15]:
r = await _get_redis()

# ════════════════════════════════════════════════════════════════════
# Step 2: 模拟后台 Worker（GET 请求触发的副作用）
# ════════════════════════════════════════════════════════════════════
async def process_one_task(task_id: str):

    # ── 给 Worker 设个超时，避免没任务时无限阻塞 ─────────────
    try:
        await asyncio.wait_for(_process_one_task(r), timeout=600)
        print(f"  → Worker 处理完毕")
    except asyncio.TimeoutError:
        print(f"  ⚠ Worker 超时（10 分钟）— 请检查 LLM 调用是否卡住")
    except Exception as exc:
        logger.exception(f"Worker 处理失败: {exc}")


await process_one_task(task_id)


2026-09-13 14:40:06 | INFO     | agent.task_queue:_process_one_task:166 - [TaskQueue] 开始执行任务 task_id=34dfae26...
2026-09-13 14:40:19 | INFO     | agent.graph:generate_plan:78 - [MainGraph] 生成的计划 (2752 字)
2026-09-13 14:40:19 | INFO     | agent.graph:evaluate_plan:100 - [MainGraph] 等待用户确认计划
2026-09-13 14:40:19 | INFO     | agent.task_queue:_process_one_task:274 - [TaskQueue] 任务完成 task_id=34dfae26...


  → Worker 处理完毕
